In [307]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [308]:
import jieba
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from torch.distributed.pipelining import pipeline
from umap import UMAP
import numpy as np
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

In [309]:
import torch
torch.cuda.is_available()
torch.__version__

'2.8.0+cu126'

In [310]:
# 导入数据
# with open('陕西-3.txt', 'r', encoding='utf-8') as f:
#     docs=f.readlines()
with open('党史数据库切词.txt', 'r', encoding='utf-8') as f:
    docs=f.readlines()
print("文本数：",len(docs))
print("第一条：",docs[0])
vectorizer_model = None

文本数： 2410
第一条： 二零一二年



In [311]:
# set HF_ENDPOINT=https://hf-mirror.com
# 0 sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# embedding_model=SentenceTransformer("C:\\Users\\xule\\.cache\\huggingface\\hub\\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2\\snapshots\\86741b4e3f5cb7765a600d3a3d55a0f6a6cb443d")
# embedding_model=SentenceTransformer("C:\\Users\\xule\\.cache\\huggingface\\hub\\models--KaLM-Embedding--KaLM-embedding-multilingual-mini-instruct-v2.5\\snapshots\\753c6fe26abc20a32aeb162003aa03457d15db2f")
from transformers import pipeline
# print(transformers.__version__)
embedding_model = pipeline("feature-extraction", model="bert-base-chinese", device="cuda")
embeddings=np.load('emb_党史数据库.npy')
# print(1)

In [312]:
umap_model=UMAP(
    n_neighbors=5,  #参考周围的5个连续点
    n_components=5, #将高维数据降维到5维
    min_dist=0.0,   #嵌入点之间的最小的有效距离，值越小会导致越密集的聚类（会减少离群值），默认为0.1
    metric='cosine',    #余弦距离，默认会使用欧几里得距离
    random_state=49 #设置随机数种子
)

In [313]:
hdbscan_model=HDBSCAN(
    min_cluster_size=25,
    min_samples=5,  #减少该值可以减少离群值，默认等于min_cluster_size
    metric='euclidean',
    # prediction_data=True,   #若计算文本属于每一个主题的概率，则必须为True
)   #每个类中至少包含5条

In [314]:
# def tokenize_zh(text):  #切词
#     words=jieba.cut(text)
#     return words
#
# # 若导入的是切词后的文件，则下一条语句无需填写参数
# vectorizer_model=CountVectorizer(tokenizer=tokenize_zh)
vectorizer_model=CountVectorizer(stop_words=['有着'])

In [315]:
topic_model=BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    # min_topic_size=5, #每个主题至少包含5条，有min_cluster_size就不用设置
    # nr_topics='auto',   #自动合并主题
    # calculate_probabilities=True,   #计算属于每一个主题的概率
)
# print(embeddings.shape)
# print(len(docs))
topics, probs=topic_model.fit_transform(docs)
topic_info=topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,686,-1_人民_历史_发展_伟大,"[人民, 历史, 发展, 伟大, 中国共产党, 革命, 中华民族, 民族, 特色, 事业]",[深人 学习 宣传 中国人民志愿军 英雄事迹 革命 学好 党史 中国史 改革开放 发展史 激...
1,0,233,0_马克思主义_邓小平_改革开放_理论,"[马克思主义, 邓小平, 改革开放, 理论, 毛泽东思想, 毛泽东, 历史, 建设, 问题,...",[改革开放 解决 正确 评价 毛泽东 毛泽东思想 历史 地位 实际 历史 经验 实现 现代化...
2,1,153,1_政治_干部_党的建设_领导,"[政治, 干部, 党的建设, 领导, 问题, 提高, 执政, 能力, 务必, 作风]",[征程 牢记 打铁 道理 从严治党 路上 政治 自觉 政治 建设 统领 党的建设 伟大工程 ...
3,2,131,2_伟大复兴_实现_中华民族_奋斗,"[伟大复兴, 实现, 中华民族, 奋斗, 各族人民, 梦想, 伟大, 光明, 长征, 前景]","[完全 中华民族 伟大复兴 实现 实现\n, 实现 中华民族 伟大复兴 中华民族 近代 伟大..."
4,3,127,3_红军_根据地_焦裕禄_革命,"[红军, 根据地, 焦裕禄, 革命, 长征, 英雄, 抗日, 延安, 烈士, 陕甘]",[杨靖宇 赵尚志 左权 彭雪枫 佟麟阁 赵登禹 张自忠 戴安澜 殉国 将领 八路军 狼牙山 ...
5,4,97,4_制度_历史_国家_大国,"[制度, 历史, 国家, 大国, 中华民族, 发展, 民族, 国民经济, 奠定, 建设]",[伟大 历史 贡献 意义 中华民族 有史以来 广泛 社会变革 当代 发展 进步 奠定 政治 ...
6,5,70,5_群众_人民_人民群众_根本利益,"[群众, 人民, 人民群众, 根本利益, 服务, 青年, 群众路线, 得到, 源泉, 发展]",[全党同志 人民 放在 心中 最高 位置 全心全意 人民 服务 根本宗旨 实现 发展 人民 ...
7,6,68,6_终结_脱离_无往而不胜_关键,"[终结, 脱离, 无往而不胜, 关键, 地方, 力量, 孔子, 所归, 一事无成, 无本之木]","[人民 脱离 人民 无源之水 无本之木 一事无成\n, 自诩 高明 脱离 人民 凌驾于 人民..."
8,7,66,7_军队_发展_国防_科技,"[军队, 发展, 国防, 科技, 创新, 经济, 改革, 体系, 建设, 强军]",[全军 深化 国防 军队 改革 解决 制约 国防 军队 建设 体制性 障碍 结构性 矛盾 政...
9,8,66,8_教育_坚定_信念_理论自信,"[教育, 坚定, 信念, 理论自信, 制度自信, 文化自信, 理想信念, 道路自信, 远大理...","[全党 坚定 道路自信 理论自信 制度自信 文化自信\n, 红军 丰功伟绩 伟大 长征 爱国..."


In [316]:
print(len(topics),topics[:10])
print(len(probs),probs[:10])

2410 [-1, 2, 4, -1, 13, -1, 10, -1, 6, 10]
2410 [0.         1.         0.84132689 0.         1.         0.
 0.94728659 0.         0.44551009 0.94728659]


In [317]:
# topic_model.set_topic_labels({
#     0:'',
#     1:'',
#     2:''
# })
# topic_info=topic_model.get_topic_info()
# topic_info
# topic_model.visualize_barchart(custom_labels=True)
topic_model.visualize_barchart()

In [318]:
topic_model.visualize_topics()

In [319]:
reduced_embeddings = UMAP(n_neighbors=5, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings, hide_document_hover=True)

In [320]:
import plotly.express as px
hierarchical_topics = topic_model.hierarchical_topics(docs)
# topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

# 为每个轨迹随机分配颜色（或使用预定义颜色列表）
colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']
for i, trace in enumerate(fig.data):
    trace.line.color = colors[i % len(colors)]

fig.show()

100%|██████████| 27/27 [00:00<00:00, 252.37it/s]


In [321]:
with open('党史数据库时间.txt','r',encoding='utf-8') as file:
    lines=file.readlines()
    timestamps=[int(line.strip()) for line in lines]
print(len(timestamps),timestamps[:10])

2410 [2012, 2012, 2012, 2012, 2012, 2012, 2012, 2012, 2012, 2012]


In [322]:
topics_over_time=topic_model.topics_over_time(docs,timestamps,global_tuning=False, evolution_tuning=False) # global_tuning=False, evolution_tuning=False
topic_model.visualize_topics_over_time(topics_over_time)

In [323]:
topic_docs = topic_model.get_document_info(docs)
topic_docs.to_csv('党史数据库聚类结果.csv')